# 1. KQL Hunting Patterns

KQL (Kusto Query Language) is the language of Sentinel, Defender XDR, Azure Monitor, and Azure Data Explorer. SC-200 requires you to write and interpret KQL queries.

## Bad hunt → good hunt

A **bad hunt** is eyeballing raw logs, hoping to spot something weird. It doesn't scale — one analyst, one screen, thousands of events per minute.

A **good hunt** uses *aggregation and filters* to reduce noise and surface outliers:

| Bad (don't do this) | Good (do this) |
|---------------------|----------------|
| Scroll through 10k raw sign-ins | `summarize count() by UserPrincipalName` |
| Search logs for "bad" by hand | `where DestinationIP in (ThreatIntel_IPs)` |
| Look for one indicator at a time | Correlate across tables (sign-in + device + firewall) |
| One-off hunt, forget it | Convert successful hunt into a **scheduled detection rule** |

## KQL basics → SIEM query mapping

Our mini-SIEM API mirrors the most important KQL concepts:

| KQL | Our SIEM API equivalent |
|-----|------------------------|
| `SigninLogs` | `table_name: 'SigninLogs'` |
| `\| where ResultType != 0` | `filter: {ResultType: 'Failure'}` |
| `\| summarize count() by UserPrincipalName` | `aggregate_by: 'UserPrincipalName'` |
| `\| where TimeGenerated > ago(1h)` | `time_range_minutes: 60` |
| `\| take 10` | `limit: 10` |

We'll write queries in both formats so you learn KQL while using the SIEM.

> ⚠️ **Where the mini-SIEM differs from the real schema — read this before the exam**
>
> | | Real Log Analytics / Defender XDR | This mini-SIEM |
> |---|---|---|
> | Sign-in outcome | `SigninLogs.ResultType` is a **string containing a numeric error code**: `"0"` = success, `"50126"` = invalid username/password, `"50053"` = account locked, `"50076"` = MFA required. `ResultDescription` holds the text. | Simplified to the strings `'Success'` / `'Failure'` |
> | Timestamp column | `TimeGenerated` (datetime) | `timestamp` (ISO string) |
> | Azure Firewall logs | Resource-specific tables `AZFWNetworkRule`, `AZFWApplicationRule`, `AZFWNatRule` (or legacy `AzureDiagnostics`) | a single flat `AzureFirewall` table |
>
> So the exam-correct way to find failed sign-ins is **`| where ResultType != 0`** (or
> `!= "0"`), *never* `== "Failure"`. Every "real KQL" block below uses the real schema;
> the API call underneath uses the mini-SIEM's simplified one.


In [ ]:
import httpx, json
from collections import Counter, defaultdict

SIEM = 'http://localhost:8000'

def query(table, filter=None, aggregate_by=None, time_range=None, limit=100):
    r = httpx.post(f'{SIEM}/query', json={
        'table_name': table,
        'filter': filter,
        'aggregate_by': aggregate_by,
        'time_range_minutes': time_range,
        'limit': limit,
    })
    return r.json()['results']

# ----- Bad hunt demo: scroll through raw logs by eye -----
print('=== BAD HUNT: eyeballing raw sign-ins (first 5 of many) ===')
raw = query('SigninLogs', limit=5)
for row in raw:
    print(f"  {row['timestamp'][:19]}  {row['UserPrincipalName']:<22} {row['ResultType']:<8} {row['IPAddress']:<16} {row['Location']}")
print('\n  → With thousands of events, this does not scale. We need aggregation.\n')


In [ ]:
# ----- Hunt 1: Who has the most failed sign-ins? -----
# MITRE ATT&CK: T1110 Brute Force (TA0006 Credential Access)
print('=== Hunt 1: Accounts with most failed sign-ins ===')
print('KQL: SigninLogs')
print('     | where TimeGenerated > ago(1d)')
print('     | where ResultType != 0                      // 0 == success; anything else failed')
print('     | summarize FailCount = count() by UserPrincipalName')
print('     | sort by FailCount desc\n')

results = query('SigninLogs', filter={'ResultType': 'Failure'}, aggregate_by='UserPrincipalName')
for r in results:
    bar = '█' * min(r['count'], 40)
    print(f'  {r["group_key"]:<30} {r["count"]:>3} {bar}')


In [ ]:
# ----- Hunt 2: Where are sign-ins coming from? -----
# MITRE ATT&CK: T1078 Valid Accounts (TA0001 Initial Access)
print('=== Hunt 2: Sign-in locations ===')
print('KQL: SigninLogs | summarize count() by Location | sort by count_ desc\n')

results = query('SigninLogs', aggregate_by='Location')
suspicious_locs = {'Moscow', 'Beijing', 'Anonymous Proxy'}
for r in results:
    flag = ' ⚠️' if r['group_key'] in suspicious_locs else ''
    print(f'  {r["group_key"]:<20} {r["count"]:>3}{flag}')


In [ ]:
# ----- Hunt 3: Rare OR known-bad processes on endpoints -----
# MITRE ATT&CK: T1059 Command & Scripting (Execution), T1003 Credential Dumping
#
# Why "rare OR known-bad"?  A threshold alone misses things: in a small lab,
# attack tools might show up a handful of times (not truly rare). A known-bad
# list catches them regardless. In real hunting you combine both signals.
print('=== Hunt 3: Rare processes + known attack tools ===')
print('KQL (process launches live in DeviceProcessEvents, not DeviceEvents):')
print('     DeviceProcessEvents')
print('     | where Timestamp > ago(7d)')
print('     | summarize Runs = count(), Hosts = dcount(DeviceName) by FileName')
print('     | where Runs <= 6 or FileName in~ ("mimikatz.exe","psexec.exe","certutil.exe")')
print('     | sort by Runs asc\n')

results = query('DeviceEvents', aggregate_by='FileName')
known_attack_tools = {'mimikatz.exe', 'psexec.exe', 'certutil.exe', 'nc.exe', 'powershell.exe'}

# Threshold of 6 works well for our small dataset; tune for real environments.
for r in sorted(results, key=lambda x: x['count']):
    name = r['group_key']
    if r['count'] <= 6 or name in known_attack_tools:
        tag = '🔴 ATTACK TOOL' if name in known_attack_tools else '🟡 rare'
        print(f'  {name:<20} seen {r["count"]:>2} time(s)  {tag}')


In [ ]:
# ----- Hunt 4: Outbound connections to unusual destinations -----
# MITRE ATT&CK: T1041 Exfiltration Over C2, T1071 Application Layer Protocol
print('=== Hunt 4: Outbound traffic analysis ===')
print('KQL (real Azure Firewall table, and filter BEFORE the summarize):')
print('     AZFWNetworkRule')
print('     | where TimeGenerated > ago(1d)')
print('     | where not(ipv4_is_private(DestinationIp))   // beats !startswith "10."')
print('     | summarize ConnectionCount = count() by DestinationIp')
print('     | sort by ConnectionCount desc\n')

results = query('AzureFirewall', aggregate_by='DestinationIP')
known_bad = {'185.220.101.42', '45.33.32.156', '198.51.100.99'}

for r in results:
    ip = r['group_key']
    if ip and not ip.startswith('10.'):
        threat = '🔴 KNOWN BAD' if ip in known_bad else '🟡 External'
        print(f'  {ip:<20} {r["count"]:>3} connections  {threat}')


In [ ]:
# ----- Hunt 5: Failures followed by success (brute-force succeeded) -----
# MITRE ATT&CK: T1110.001 Password Guessing + T1078 Valid Accounts
#
# This is a classic "join" hunt — in real KQL you'd `join` two summarized
# tables. Our simple API doesn't support joins, so we join in Python.
# The lesson is the *pattern*, not the language.
print('=== Hunt 5: Users with many failures AND at least one success ===')
print('KQL equivalent:')
print('  let failures  = SigninLogs | where ResultType != 0 | summarize FailCount=count()    by UserPrincipalName;')
print('  let successes = SigninLogs | where ResultType == 0 | summarize SuccessCount=count() by UserPrincipalName;')
print('  failures')
print('  | join kind=inner successes on UserPrincipalName   // kind= is NOT optional here!')
print('  | where FailCount > 5')
print('  | project UserPrincipalName, FailCount, SuccessCount\n')

all_signins = query('SigninLogs', limit=1000)
user_stats = defaultdict(lambda: {'failures': 0, 'successes': 0, 'ips': set()})

for s in all_signins:
    user = s['UserPrincipalName']
    if s['ResultType'] == 'Failure':
        user_stats[user]['failures'] += 1
    else:
        user_stats[user]['successes'] += 1
    user_stats[user]['ips'].add(s['IPAddress'])

for user, stats in user_stats.items():
    if stats['failures'] > 5 and stats['successes'] > 0:
        print(f'  🔴 {user}: {stats["failures"]} failures + {stats["successes"]} successes')
        print(f'     IPs: {sorted(stats["ips"])}')
        print(f'     → Likely COMPROMISED — investigate immediately!\n')


## Real KQL operators for the SC-200 exam

| Operator | What it does | Example |
|----------|-------------|----------|
| `where` | Filter rows | `\| where ResultType != 0`  ← real `SigninLogs` schema |
| `summarize` | Aggregate | `\| summarize count() by UserPrincipalName` |
| `project` | Select columns | `\| project TimeGenerated, User, IP` |
| `extend` | Add computed column | `\| extend Hour = hourofday(TimeGenerated)` |
| `join` | Combine tables — **always state `kind=`** (see the table below; the default is `innerunique`, not `inner`) | `T1 \| join kind=inner T2 on UserId` |
| `union` | Stack tables vertically | `SigninLogs \| union DeviceLogonEvents` |
| `sort by` | Order results | `\| sort by count_ desc` |
| `top` | First N rows | `\| top 10 by count_` |
| `ago()` | Time filter | `\| where TimeGenerated > ago(1h)` |
| `bin()` | Time bucketing | `bin(TimeGenerated, 5m)` |
| `make_series` | Time series | For baselines & anomaly detection |
| `render` | Visualization | `\| render timechart` |
| `let` | Named expression: a scalar, a list, **or a whole table** | `let threshold = 5;` / `let vips = dynamic(["alice","bob"]);` / `let recent = SigninLogs \| where TimeGenerated > ago(1h);` |
| `materialize()` | Cache a `let`-bound tabular expression so it is computed **once** even if referenced many times | `let base = materialize(SigninLogs \| where TimeGenerated > ago(1d));` |
| `arg_max()` / `arg_min()` | Return the **whole row** where a column is max/min — the standard "latest record per entity" idiom | `\| summarize arg_max(TimeGenerated, *) by DeviceName` |
| `externaldata` | Load external data | CSV, JSON files |
| `mv-expand` | Expand arrays | Unpack multi-value fields |
| `dcount()` / `make_set()` | Distinct count / collect values into an array | `\| summarize dcount(DeviceName), make_set(FileName) by AccountName` |
| `ipv4_is_private()` / `ipv4_is_in_range()` | Correct IP handling instead of string prefixes | `\| where not(ipv4_is_private(RemoteIP))` |
| `hourofday()` / `dayofweek()` | Time-of-day / day-of-week extraction | `\| extend Hour = hourofday(TimeGenerated)` |

### `join` kinds — the single most commonly missed KQL fact

```kusto
LeftTable | join kind=<kind> [hint.strategy=broadcast] (RightTable) on Key
```

| `kind=` | Rows returned |
|---|---|
| **`innerunique`** ⚠️ | **THE DEFAULT if you omit `kind=`.** Silently **de-duplicates the LEFT table on the join key first**, then does an inner join. |
| `inner` | Standard SQL inner join: every matching left row × every matching right row |
| `leftouter` | All left rows; right columns null where there is no match |
| `rightouter` / `fullouter` | Mirror / union of the two outer joins |
| `leftsemi` | Left rows **that have** a match — left columns only (an existence filter) |
| `leftanti` ⭐ | Left rows **with no** match — the "first time we have ever seen this" hunting idiom |
| `rightsemi` / `rightanti` | Mirror of the two above |

> **The trap**: `T1 | join T2 on Key` is **not** an inner join — it is `innerunique`, and it
> throws away duplicate left-hand rows *before* joining. If your counts come out mysteriously
> low, you forgot `kind=inner`. Always write the `kind=` explicitly.
>
> **The hunting workhorse** is `leftanti`: "processes seen today that were never seen in the
> previous 30 days", "users signing in from a country they have never used before".

```kusto
// leftanti: hosts running a process for the FIRST time in 30 days
let historic = DeviceProcessEvents
    | where Timestamp between (ago(30d) .. ago(1d))
    | distinct DeviceName, FileName;
DeviceProcessEvents
| where Timestamp > ago(1d)
| distinct DeviceName, FileName
| join kind=leftanti historic on DeviceName, FileName

// arg_max: the latest known state of every device, one row each
DeviceInfo
| summarize arg_max(Timestamp, *) by DeviceId

// materialize: compute the expensive base set once, use it three times
let base = materialize(SigninLogs | where TimeGenerated > ago(7d));
let fails = base | where ResultType != 0 | summarize F = count() by UserPrincipalName;
let oks   = base | where ResultType == 0 | summarize S = count() by UserPrincipalName;
fails | join kind=inner oks on UserPrincipalName | where F > 5
```

### Key KQL tables for SC-200

| Table | Product | Contains |
|-------|---------|----------|
| `SigninLogs` | Entra ID | User sign-in events |
| `AuditLogs` | Entra ID | Directory changes |
| `DeviceProcessEvents` | Defender for Endpoint | **Process creation** — the table for "what ran" |
| `DeviceFileEvents` | Defender for Endpoint | File create / modify / delete |
| `DeviceRegistryEvents` | Defender for Endpoint | Registry key & value changes |
| `DeviceEvents` | Defender for Endpoint | *Miscellaneous* / catch-all events (AMSI, WMI, screenshot, USB…) — **not** where process launches live |
| `DeviceNetworkEvents` | Defender for Endpoint | Network connections |
| `DeviceLogonEvents` | Defender for Endpoint | Interactive / network logons on devices |
| `EmailEvents` | Defender for Office 365 | Email delivery |
| `UrlClickEvents` | Defender for Office 365 | Safe Links clicks |
| `CloudAppEvents` | Defender for Cloud Apps | SaaS activity |
| `IdentityLogonEvents` | Defender for Identity | On-prem AD logons |
| `SecurityAlert` | Sentinel | All alerts, from every connected product |
| `SecurityIncident` | Sentinel | All incidents |
| `AlertEvidence` | Defender XDR | The entities (file, IP, user, device) attached to each alert |
| `IdentityInfo` | Sentinel UEBA | Entra user snapshot: roles, manager, department — the standard enrichment join |
| `BehaviorAnalytics` | Sentinel UEBA | Scored activity with an investigation priority |
| `AADNonInteractiveUserSignInLogs` | Entra ID | Token-refresh / service sign-ins — **easy to miss and where a lot of attacker activity hides** |

**Next**: [Notebook 2 — Advanced Threat Hunting](02_advanced_hunting.ipynb)


---
## ✅ Self-check

1. Write the KQL filter for "failed Entra sign-ins" against the **real** `SigninLogs` schema.
   Why is `where ResultType == "Failure"` wrong?
2. `Failures | join Successes on UserPrincipalName` returns fewer rows than you expect.
   What is the cause and the one-word fix?
3. Which `join kind` answers "which processes ran today that we have never seen in the last
   30 days"?
4. You need the most recent record for each device, with all its columns. Which function?
5. Your query references the same expensive filtered table three times. What do you wrap it in?
6. Where do process-creation events live in Defender XDR — `DeviceEvents` or
   `DeviceProcessEvents`?
7. Your hunt found something real. What is the last step of the hunting cycle?

In [ ]:
answers = """
1. SigninLogs | where ResultType != 0            (or != "0")
   ResultType is the Entra sign-in ERROR CODE as a string: "0" means success, and
   everything else is a specific failure reason (50126 invalid credentials, 50053
   smart lockout, 50076 MFA required, 53003 blocked by Conditional Access...).
   There is no "Failure" value in the schema, so `== "Failure"` matches nothing and
   your rule silently never fires. Use ResultDescription if you want the text.

2. The DEFAULT join kind is INNERUNIQUE, not inner. It de-duplicates the LEFT table on
   the join key before joining, so duplicate left rows are silently dropped.
   Fix: kind=inner (write kind= explicitly, always).

3. kind=leftanti -- return left rows that have NO match on the right. It is the
   canonical "first time ever seen / rare in environment" hunting pattern.
   (kind=leftsemi is the mirror: rows that DO have a match, left columns only.)

4. arg_max(): `| summarize arg_max(Timestamp, *) by DeviceId` returns the entire row
   with the largest Timestamp per device. Using max(Timestamp) would give you only the
   timestamp and lose the rest of the row.

5. materialize(). `let base = materialize(<expensive expression>);` caches the result
   for the lifetime of the query, so the source is scanned once instead of three times.
   It only helps when the let-bound expression is referenced more than once.

6. DeviceProcessEvents. DeviceEvents is the miscellaneous/catch-all table (AMSI, WMI,
   USB, screenshots, and other event types). Pointing a process hunt at DeviceEvents is
   a common and quiet mistake.

7. Operationalise it: turn the query into a SCHEDULED ANALYTICS RULE (Sentinel) or a
   CUSTOM DETECTION RULE (Defender XDR), and attach an automation rule/playbook. A hunt
   you have to remember to re-run is not a control.
"""
print(answers)